# ACCA Translate — dịch tài liệu ACCA thành bản song ngữ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trinhten1309-cyber/VI-Translate/blob/main/colab/ACCA_Translate.ipynb)

Toàn bộ code nằm trong repo [trinhten1309-cyber/VI-Translate](https://github.com/trinhten1309-cyber/VI-Translate) (thư mục `colab/`). Notebook này chỉ cài đặt và gọi code đó, nên **mỗi lần chạy luôn dùng bản mới nhất** — không cần tải lại notebook khi repo được sửa.

| Phần | Việc làm |
|---|---|
| 1 | Kết nối Google Drive |
| 2 | Cài repo + thư viện, gồm cả OCR (~2–3 phút) |
| 3 | Nạp API key + kiểm tra từng nguồn |
| 4 | **Dịch PDF** → bản dịch + bản song ngữ trong `output/` |

**Thành viên mới (làm một lần):**
1. Thêm shortcut thư mục dùng chung vào My Drive (Shared with me → chuột phải → Organize → Add shortcut → My Drive).
2. Lấy API key Gemini, Groq, OpenRouter bằng tài khoản của mình và lưu vào 🔑 Secrets (hướng dẫn ở phần 3).
3. **File → Save a copy in Drive** để lưu lại lựa chọn trong form.

**Mỗi lần dùng:** upload PDF vào `input/` → **Runtime → Run all** → đổi file/trang ở phần 4 và chạy lại riêng ô đó.

## 1. Kết nối Google Drive
`FOLDER` là tên thư mục dùng chung nằm trong My Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob
FOLDER = "DOCUMENTS TRANSLATION"  # @param {type:"string"}
BASE = f"/content/drive/MyDrive/{FOLDER}"
assert os.path.isdir(BASE), f"Không thấy thư mục {BASE} — kiểm tra tên thư mục hoặc shortcut trong My Drive"
for sub in ["input", "output", "glossary", "cache"]:
    os.makedirs(f"{BASE}/{sub}", exist_ok=True)

print("Thư mục làm việc:", BASE)
print("PDF trong input/:", [os.path.basename(p) for p in sorted(glob.glob(f"{BASE}/input/*.pdf"))] or "⚠️ chưa có — hãy upload PDF vào input/")
print("Bảng thuật ngữ:", [os.path.basename(p) for p in glob.glob(f"{BASE}/glossary/*.csv")] or "chưa có (vẫn dịch được)")

## 2. Cài repo và thư viện
Repo cần Python 3.12 (thư viện `babeldoc` chưa hỗ trợ Python 3.13 của Colab), nên được cài trong môi trường riêng `.venv`.

In [ ]:
REPO_URL = "https://github.com/trinhten1309-cyber/VI-Translate.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
os.environ["REPO_URL"], os.environ["BRANCH"] = REPO_URL, BRANCH

In [ ]:
%%bash
set -e
cd /content
if [ -d VI-Translate/.git ]; then
  git -C VI-Translate fetch -q --depth 1 origin "$BRANCH" && git -C VI-Translate checkout -q -B "$BRANCH" FETCH_HEAD
else
  git clone -q --depth 1 --branch "$BRANCH" "$REPO_URL" VI-Translate
fi
command -v uv > /dev/null || pip install -q uv
hash -r
UV=$(command -v uv)
cd VI-Translate
[ -d .venv ] || "$UV" venv -q -p 3.12 .venv
"$UV" pip install -q -p .venv/bin/python -r requirements-ocr.txt openai
command -v tesseract > /dev/null || { apt-get -qq update > /dev/null && apt-get -qq install -y tesseract-ocr > /dev/null; }
test -f colab/acca_translate.py || { echo "❌ Nhánh $BRANCH chưa có thư mục colab/"; exit 1; }
echo "✅ Cài đặt xong — phiên bản $(git log -1 --format='%h %cd' --date=short)"

## 3. API key

| Nhà cung cấp | Nơi lấy key | Tên secret |
|---|---|---|
| Gemini | aistudio.google.com/apikey → Create API key | `GEMINI_API_KEY` |
| Groq | console.groq.com/keys → Create API Key | `GROQ_API_KEY` |
| OpenRouter | openrouter.ai/settings/keys → Create API key | `OPENROUTER_API_KEY` |

- Groq và OpenRouter chỉ hiện key đầy đủ **một lần** lúc tạo — copy ngay trong hộp thoại. OpenRouter: chọn Expiration bất kỳ và Credit limit nhỏ nhất.
- **Không nhập thẻ thanh toán ở bất kỳ trang nào.** Không bật billing cho project Gemini.
- **Lưu key:** biểu tượng 🔑 (cột trái) → **Add new secret** → điền đúng tên ở cột cuối → dán key vào Value → bật **Notebook access**.
- `ORDER`: nguồn đứng trước dùng trước; hết lượt thì tự chuyển sang nguồn sau.

**Kết quả kiểm tra:** ✅ tốt · ⚠️ trả lời được nhưng sai thẻ/thuật ngữ (vẫn dùng được) · ❌ `401` key sai, `404` tên mô hình đã đổi (điền tên mới vào form), "hết hạn mức trong ngày" = hết lượt free hôm nay · ⏭️ chưa có key / sai tên secret / chưa bật Notebook access.

In [ ]:
from google.colab import userdata

ORDER = "gemini,groq,openrouter"  # @param {type:"string"}
GEMINI_MODEL = "gemini-3.5-flash"  # @param {type:"string"}
GROQ_MODEL = "openai/gpt-oss-120b"  # @param {type:"string"}
OPENROUTER_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"  # @param {type:"string"}

os.environ["PROVIDER_ORDER"] = ORDER
for name, model in [("GEMINI", GEMINI_MODEL), ("GROQ", GROQ_MODEL), ("OPENROUTER", OPENROUTER_MODEL)]:
    os.environ[f"{name}_MODEL"] = model
    try:
        os.environ[f"{name}_API_KEY"] = userdata.get(f"{name}_API_KEY")
    except Exception:
        os.environ.pop(f"{name}_API_KEY", None)

PY = "/content/VI-Translate/.venv/bin/python"
RUNNER = "/content/VI-Translate/colab/acca_translate.py"
!PYTHONUNBUFFERED=1 {PY} {RUNNER} test

## 4. Dịch PDF

| Ô | Ý nghĩa |
|---|---|
| `PDF_NAME` | Tên file trong `input/`. Để trống = file đầu tiên |
| `PAGES` | Ví dụ `1-5`, `1,3,8-10`. Để trống = cả file. **Nên chạy thử 5 trang trước** |
| `STYLE` | `Cabin` (giống Canva) · `CabinSemiCondensed` (hẹp hơn, hợp tài liệu nhiều bảng) · `default` (font mặc định của repo) |
| `BILINGUAL` | `side-by-side` trang gốc và trang dịch cạnh nhau · `alternating` xen kẽ · `layers` bản gốc ở lớp ẩn · `off` chỉ bản dịch |
| `MON` | Lọc bảng thuật ngữ theo môn (F5, F7…). Để trống = dùng tất cả |
| `BATCH` | Số đoạn mỗi lượt API. Giảm xuống 10 nếu hay lỗi định dạng hoặc lỗi 413 |
| `FORCE` | Dịch lại từ đầu, bỏ qua bộ nhớ đệm (ví dụ sau khi sửa bảng thuật ngữ) |
| `OCR` | `auto`: trang **chỉ có ảnh** (scan, không bôi đen được chữ) được nhận dạng chữ rồi dựng lại thành chữ thật — dịch được cả bảng; mỗi trang thêm vài giây · `engine`: OCR thận trọng của repo, chỉ dịch đoạn văn an toàn · `off`: tắt |
| `OCR_CHECK` | Bật (nên để bật): mọi số trên trang scan được đọc lần hai bằng Tesseract; số nào hai bộ đọc ra khác nhau hoặc sai định dạng (thiếu ngoặc, `45.400` thay vì `45,400`) được liệt kê trong `<tên>-kiem-tra-ocr.pdf` để xem lại bằng mắt — không tự chọn hộ. Thêm khoảng 1 giây mỗi trang |
| `OCR_READER` | `rapid`: RapidOCR (nhanh, chạy trên CPU) · `paddle-vl`: PaddleOCR-VL đọc lại từng dòng — chính xác hơn (đọc đúng `30`, `$'000` mà RapidOCR hay đọc sai) nhưng **cần GPU**: **Runtime → Change runtime type → T4 GPU** rồi Run all. Lần đầu mỗi phiên mất vài phút cài đặt và tải mô hình. Không có GPU thì tự quay về `rapid` |
| `IMAGE_PDF` | `auto`: PDF là ảnh chụp/scan có lớp chữ OCR ẩn thì tự xoá chữ trong ảnh, dựng lại thành chữ thật trước khi dịch (tránh chữ dịch đè lên chữ gốc) · `off`: tắt |

**Kết quả trong `output/`:** `<tên>-vi.pdf` (bản dịch), `<tên>-vi-dual.pdf` (song ngữ), `<tên>-can-xem-lai.jsonl` nếu có đoạn không qua kiểm tra, và `<tên>-kiem-tra-ocr.pdf` nếu có số trên trang scan cần xem lại (khung đỏ đánh số trên trang gốc, bên phải là cách đọc của từng bộ đọc). Cả hai file PDF **chỉ gồm các trang trong `PAGES`** — để trống `PAGES` khi muốn dịch cả file. Hết lượt API giữa chừng → chạy lại ô này (hôm sau nếu hết lượt ngày); phần đã dịch không bị dịch lại.

**Giới hạn của engine:** trang mục lục / chỉ mục câu hỏi (bảng có số trang cuối dòng) được giữ nguyên, không dịch — cảnh báo "image-only scans" cho các trang này là bình thường. Bảng không có đường kẻ hoặc cột căn thẳng có thể chỉ được dịch một phần. Với `OCR = auto`, dòng nào nhận dạng không chắc chắn được giữ nguyên dạng ảnh (còn tiếng Anh) thay vì bị thay bằng chữ sai; ảnh scan mờ, nghiêng hoặc chữ viết tay sẽ còn nhiều dòng như vậy.

In [ ]:
PDF_NAME = ""  # @param {type:"string"}
PAGES = "1-5"  # @param {type:"string"}
STYLE = "Cabin"  # @param ["Cabin", "CabinSemiCondensed", "default"]
BILINGUAL = "side-by-side"  # @param ["side-by-side", "alternating", "layers", "off"]
MON = ""  # @param {type:"string"}
BATCH = 15  # @param {type:"integer"}
FORCE = False  # @param {type:"boolean"}
OCR = "auto"  # @param ["auto", "engine", "off"]
IMAGE_PDF = "auto"  # @param ["auto", "off"]
OCR_CHECK = True  # @param {type:"boolean"}
OCR_READER = "rapid"  # @param ["rapid", "paddle-vl"]

pdfs = sorted(glob.glob(f"{BASE}/input/*.pdf"))
assert pdfs, f"Chưa có PDF trong {BASE}/input/"
pdf = f"{BASE}/input/{PDF_NAME}" if PDF_NAME else pdfs[0]
assert os.path.exists(pdf), f"Không thấy {pdf}"
FORCE_FLAG = "--force" if FORCE else ""
CHECK = "on" if OCR_CHECK else "off"

if OCR == "auto" and OCR_READER == "paddle-vl":
    import shutil, subprocess
    if not shutil.which("nvidia-smi"):
        print("⚠️  Chưa bật GPU: Runtime → Change runtime type → T4 GPU, rồi Run all lại. Lần này dùng RapidOCR.")
    elif subprocess.run([PY, "-c", "import torch, transformers"], capture_output=True).returncode:
        print("Cài PaddleOCR-VL (lần đầu mỗi phiên, khoảng 2–3 phút)...")
        UV = shutil.which("uv") or "uv"
        !{UV} pip install -q -p {PY} torch torchvision --index-url https://download.pytorch.org/whl/cu126
        !{UV} pip install -q -p {PY} transformers==4.57.6 sentencepiece

!PYTHONUNBUFFERED=1 {PY} {RUNNER} run --base "{BASE}" --pdf "{pdf}" --pages "{PAGES}" --style {STYLE} --bilingual {BILINGUAL} --batch {BATCH} --mon "{MON}" --ocr {OCR} --image-pdf {IMAGE_PDF} --ocr-check {CHECK} --ocr-reader {OCR_READER} {FORCE_FLAG}

## Bảng thuật ngữ (tùy chọn)

Đặt một hoặc nhiều file `.csv` (UTF-8) vào thư mục `glossary/`. Notebook tự đọc mỗi lần dịch và chỉ gửi kèm những thuật ngữ xuất hiện trong từng lô.

| Cột | Bắt buộc | Nội dung |
|---|---|---|
| `en` | ✓ | Thuật ngữ tiếng Anh |
| `vi` | ✓ | Thuật ngữ tiếng Việt chuẩn |
| `mon` | | F5, F7…; nhiều môn cách nhau bằng `;`; `chung` = mọi môn |
| `keep_en` | | `1` = giữ nguyên tiếng Anh (viết tắt, tên chuẩn mực) |
| `ghi_chu` | | Ghi chú cho người duyệt, không gửi cho API |

Từ Google Sheet: **File → Download → Comma-separated values (.csv)** rồi đặt vào `glossary/`. Sửa bảng thuật ngữ xong thì bật `FORCE` để dịch lại các đoạn cũ.